# 02 — Classical baseline (Anadi)

In [ ]:
# Dev notebook: imports the shared code from the repo.
# Local: run from the repo (notebooks/ folder). Kaggle: attach your private code dataset and set CODE_DIR.
import os, sys
CODE_DIR = os.environ.get('SCAMSHIELD_CODE_DIR') or ('/kaggle/input/scamshield-code' if os.path.isdir('/kaggle/input/scamshield-code') else os.path.abspath('..'))
sys.path.insert(0, CODE_DIR)
from common.data_loader import load_config, load_train, load_test, get_targets, test_available, output_dirs
from common.folds import load_folds, fold_fingerprint
from common.seed import set_seed
cfg = load_config(os.path.join(CODE_DIR, 'configs', 'base.yaml'))
SEED = set_seed(cfg['seed'])
train = load_train(cfg)
fold = load_folds(train)            # shared manifest: folds/folds.csv
print(train.shape, 'folds', sorted(set(fold)), 'test available:', test_available(cfg))

In [ ]:
from common.stress import run_stress
from common.metrics import composite_proxy
from models.classical.tfidf_lr import TfidfLR, TfidfLRConfig, make_fit_predict
mcfg = TfidfLRConfig(seed=SEED)
report, oof = run_stress(make_fit_predict(mcfg), train, seed=SEED, repeats=True)
m = report['main']
comp = composite_proxy({'F_A': m['F_A'], 'F_B': m['F_B'], 'F_C': m['F_C'], 'Calibration': m['Calibration'],
                        'F_robust': report['F_robust_proxy'], 'F_novel': report['F_novel_proxy'],
                        'F_domain': report['F_domain_proxy']})
print('OOF (grouped CV):', {k: m[k] for k in ('F_A', 'F_B', 'F_C', 'Brier', 'Calibration')})
print('Local proxies (NOT Kaggle scores):', {k: report[k] for k in ('F_robust_proxy', 'F_novel_proxy', 'F_domain_proxy')})
print('Repeated grouped CV F_A:', report['repeated_F_A'])
print('LOCAL composite proxy:', comp['composite_proxy'])

In [ ]:
import pandas as pd
from common.labels import TASK_A_NAMES
print('Per-class Task A (OOF):')
print(pd.DataFrame(m['per_class_A']).T)
print('Confusion matrix Task A (rows = true, cols = predicted):')
print(pd.DataFrame(m['confusion_A'], index=list(TASK_A_NAMES.values()), columns=range(6)))
print('Per-technique Task B (OOF):')
print(pd.DataFrame(m['per_technique_B']).T)
print('Slices (script / conversation):')
print(pd.DataFrame(report['slices_A']).T)